In [ ]:
%xmode Context
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

# help locating the package sgpykit (comment out if you installed it)
import sys
from pathlib import Path
PKG_PARENT = Path().resolve().parent
sys.path.insert(0, str(PKG_PARENT))

# load sgpykit
import sgpykit as sg

# Set up logging with sgpykit convenience functions
sg.set_logger_basic_format()
sg.set_logger_info_level()
# For more verbose output, use: sg.set_logger_debug_level()
# For custom format, use: sg.set_logger_custom_format(format)

# PART 2: Evaluate a Function on a Sparse Grid

## Basics

The kit comes with the function [`evaluate_on_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.evaluate_on_sparse_grid), which allows evaluating a function on the points of a sparse grid and provides recycling of previous evaluations.
It works for both scalar-valued and vector-valued functions. Sparse grids passed as input must be reduced.

In [ ]:
fs = lambda x: sum(x)  # or np.sum(x,axis=0)
fv = lambda x: 2 * x

N = 2
w = 3
S, _ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n, -1, 1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
# plain use of evaluate_on_sparse_grid: no recycling
evals_plain_fs, *_ = sg.evaluate_on_sparse_grid(fs, S=None, Sr=Sr)
evals_plain_fv, *_ = sg.evaluate_on_sparse_grid(fv, S=None, Sr=Sr)

In [ ]:
# a direct computation
pts = Sr.size

os = 1  # np.shape(fs(Sr.knots[:,0])[0]  # fs returns a number, so np.shape makes no sense here
ov = np.shape(fv(Sr.knots[:, 0]))[0]

In [ ]:
evals_direct_fs = np.zeros((os, pts))
evals_direct_fv = np.zeros((ov, pts))

for i in range(pts):
    evals_direct_fs[:, i] = fs(Sr.knots[:, i])
    evals_direct_fv[:, i] = fv(Sr.knots[:, i])

assert np.allclose(evals_plain_fs, evals_direct_fs), "Results Mismatch"
assert np.allclose(evals_plain_fv, evals_direct_fv), "Results Mismatch"

## Use Recycling Feature

In [ ]:
f = lambda x: sum(x)

N = 2
w = 3

S, _ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n, -1, 1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

w = 4
T, _ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n, -1, 1), sg.lev2knots_lin)
Tr = sg.reduce_sparse_grid(T)

In [ ]:
evals_non_rec, *_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Tr)
evals_old, *_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)
evals_rec, *_ = sg.evaluate_on_sparse_grid(
    f,
    T,
    Sr=Tr,
    evals_old=evals_old,
    S_old=S,
    Sr_old=Sr,
)
assert np.allclose(evals_non_rec, evals_rec), "Results Mismatch"

## Recycle from a "List of Points"

It is also possible to recycle from a list of points. However, the algorithm used to detect
which points are to be evaluated is much slower than the previous case for large $N$.

In [ ]:
f = lambda x: sum(x)

N = 20
w = 1

S, _ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n, -1, 1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

w = 2
T, _ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n, -1, 1), sg.lev2knots_lin)
Tr = sg.reduce_sparse_grid(T)

evals_non_rec, *_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Tr)
with sg.misc.time_block():
    evals_old, *_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)
    evals_rec, *_ = sg.evaluate_on_sparse_grid(
        f,
        T,
        Sr=Tr,
        evals_old=evals_old,
        S_old=S,
        Sr_old=Sr,
    )


In [ ]:
# pretend we only know the list of points Sr.knots, to see the difference in performance ...
with sg.misc.time_block():
    evals_old, *_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)
    evals_rec_slow, *_ = sg.evaluate_on_sparse_grid(
        f,
        T,
        Sr=Tr,
        evals_old=evals_old,
        S_old=S,
        Sr_old=Sr,
    )

assert np.allclose(evals_non_rec, evals_rec), "Results Mismatch"

## Use Recycling Feature for Vector Output

In [ ]:
f = lambda x: 2 * x

N = 2
w = 1

S, _ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n, -1, 1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

w = 2
T, _ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n, -1, 1), sg.lev2knots_lin)
Tr = sg.reduce_sparse_grid(T)

In [ ]:
evals_non_rec, *_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Tr)
evals_old, *_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)
evals_rec, *_ = sg.evaluate_on_sparse_grid(
    f,
    T,
    Sr=Tr,
    evals_old=evals_old,
    S_old=S,
    Sr_old=Sr,
)
assert np.allclose(evals_non_rec, evals_rec), "Results Mismatch"

## NEXT: Integration on Sparse Grids

=> **[Go to tutorial "Integration"](02-integration.ipynb)**